In [ ]:
# script takes in a data set of span_ids with span energies, span length, # measurements used per span, concurrency for that span, and the desired accuracy
# where accuracy is a % deviation from the median energy for the span length, which is treated as the ground truth
# which it calculates for the span_id
#
# builds and compares accuracy models (start with linear, and RF) based on the data such that for a given set of data it returns 
# the target # of measurements per span for that desired accuracy
# the "best performing" model of those used 'linear', 'randomforest', ... (use R^2 and MSE?)
# and the hyperparameters for that model, e.g. b0, b1, b2 for a multilinear regression - in a json file
#
# maybe it should return a 0 if it is not possible to achieve desired accuracy level?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

import statsmodels.api as sm

from scipy import stats

#from pypalettes import load_palette

sns.set_theme(style="whitegrid")

fs=24

In [15]:
# read in the data for now - will be argument later

#df_spans = pd.read_csv("../../../Data/E2EGHG/span_300.csv")

df_spans = pd.read_csv("../../../Data/E2EGHG/span_concurrent.csv")
df_spans.columns, df_spans.describe()

(Index(['trace.id', 'span.id', 'measurements.count', 'start_time_us',
        'energy.joules', 'start.energy.joules', 'end.energy.joules',
        'kepler.duration.ms', 'kepler.energy.joules', 'exper_name', 'cpu_ms',
        'active_interval_ms', 'concurrency'],
       dtype='str'),
        measurements.count  start_time_us  energy.joules  start.energy.joules  \
 count              3221.0   3.221000e+03    3221.000000          3221.000000   
 mean                  2.0   1.776710e+15       7.226127             0.349183   
 std                   0.0   1.687559e+09       2.691560             0.308351   
 min                   2.0   1.776707e+15       0.030025             0.001245   
 25%                   2.0   1.776709e+15       5.087804             0.245756   
 50%                   2.0   1.776710e+15       6.580615             0.287645   
 75%                   2.0   1.776712e+15       8.719105             0.348900   
 max                   2.0   1.776713e+15      16.750145            

In [16]:
df_spans.columns

Index(['trace.id', 'span.id', 'measurements.count', 'start_time_us',
       'energy.joules', 'start.energy.joules', 'end.energy.joules',
       'kepler.duration.ms', 'kepler.energy.joules', 'exper_name', 'cpu_ms',
       'active_interval_ms', 'concurrency'],
      dtype='str')

In [17]:
# replace . with _ in column names
df_spans.columns = df_spans.columns.str.replace('.', '_')
df_spans.columns

Index(['trace_id', 'span_id', 'measurements_count', 'start_time_us',
       'energy_joules', 'start_energy_joules', 'end_energy_joules',
       'kepler_duration_ms', 'kepler_energy_joules', 'exper_name', 'cpu_ms',
       'active_interval_ms', 'concurrency'],
      dtype='str')

## Extract expected input columns

In [18]:
# create reduced dataset for modeling
df = df_spans.drop([ 'trace_id', 'start_time_us','start_energy_joules', 'end_energy_joules',
       'kepler_duration_ms', 'kepler_energy_joules', 'exper_name', 'active_interval_ms'], axis=1)
df.head(10)

,span_id,measurements_count,energy_joules,cpu_ms,concurrency
0,83bd5bbc03d550e9,2,9.614202,1500,9.902088
1,53481f52b03557bd,2,9.412002,1500,9.799933
2,2adb0c3f36bec31b,2,9.739511,1500,9.915722
3,8238992843699b54,2,11.120745,1500,9.951648
4,450ad86976e1cd6b,2,7.571902,1500,9.599694
5,b9ee559795f0ace3,2,10.823362,1500,9.797462
6,2de0b0ed90f6ac16,2,10.752869,1500,9.786745
7,88403fee0f8b8ae6,2,6.648150,1500,9.815419
8,d50d5481523a03bb,2,7.145428,1500,9.831703
9,47fc297f60ede9e9,2,7.374982,1500,9.704358


In [19]:
np.unique(df.span_id).shape[0], df.shape[0]

(3221, 3221)

## Now adding accuracy column for modeling

### Get Median E for each cpu_ms

In [20]:
df_median_e_cpu = df[['energy_joules','cpu_ms']].copy()
df_median_e_cpu = df_median_e_cpu.groupby(['cpu_ms'], as_index=False).median()
df_median_e_cpu

,cpu_ms,energy_joules
0,1500,6.580615


### add column with median energy for that cpu_ms to each trace_id

In [21]:
cpu_ms_vals = np.unique(df['cpu_ms'])
for i in cpu_ms_vals:
    df.loc[df['cpu_ms'] == i, 'median_energy_cpu_ms'] = df_median_e_cpu[df_median_e_cpu['cpu_ms'] == i].energy_joules.values[0]

df.head(10)

,span_id,measurements_count,energy_joules,cpu_ms,concurrency,median_energy_cpu_ms
0,83bd5bbc03d550e9,2,9.614202,1500,9.902088,6.580615
1,53481f52b03557bd,2,9.412002,1500,9.799933,6.580615
2,2adb0c3f36bec31b,2,9.739511,1500,9.915722,6.580615
3,8238992843699b54,2,11.120745,1500,9.951648,6.580615
4,450ad86976e1cd6b,2,7.571902,1500,9.599694,6.580615
5,b9ee559795f0ace3,2,10.823362,1500,9.797462,6.580615
6,2de0b0ed90f6ac16,2,10.752869,1500,9.786745,6.580615
7,88403fee0f8b8ae6,2,6.648150,1500,9.815419,6.580615
8,d50d5481523a03bb,2,7.145428,1500,9.831703,6.580615
9,47fc297f60ede9e9,2,7.374982,1500,9.704358,6.580615


### calculate deviation per trace E from median for that cpu_ms

In [22]:
df['energy_devation'] = (df.median_energy_cpu_ms - df.energy_joules) 
df

,span_id,measurements_count,energy_joules,cpu_ms,concurrency,median_energy_cpu_ms,energy_devation
0,83bd5bbc03d550e9,2,9.614202,1500,9.902088,6.580615,-3.033587
1,53481f52b03557bd,2,9.412002,1500,9.799933,6.580615,-2.831387
2,2adb0c3f36bec31b,2,9.739511,1500,9.915722,6.580615,-3.158896
3,8238992843699b54,2,11.120745,1500,9.951648,6.580615,-4.540130
4,450ad86976e1cd6b,2,7.571902,1500,9.599694,6.580615,-0.991286
...,...,...,...,...,...,...,...
3216,fb8dcc0fd37da388,2,9.964565,1500,8.880553,6.580615,-3.383950
3217,4556499a052a931f,2,13.718119,1500,3.370890,6.580615,-7.137504
3218,3a5697d1bc6df097,2,10.593429,1500,8.746059,6.580615,-4.012814
3219,e9c17bbaf0165635,2,11.990226,1500,8.765720,6.580615,-5.409611


### and add columns for % deviation from median for that cpu_ms, sampling ratio, and accuracy
#### an accuracy of 1.0 would mean it got the span mean energy value exactly for that trace

In [24]:
df['energy_deviation_pct'] = 100 * df.energy_devation / df.median_energy_cpu_ms
df['energy_deviation_pct_abs'] = np.abs(df.energy_deviation_pct)

In [25]:
df


,span_id,measurements_count,energy_joules,cpu_ms,concurrency,median_energy_cpu_ms,energy_devation,energy_deviation_pct,energy_deviation_pct_abs
0,83bd5bbc03d550e9,2,9.614202,1500,9.902088,6.580615,-3.033587,-46.098828,46.098828
1,53481f52b03557bd,2,9.412002,1500,9.799933,6.580615,-2.831387,-43.026178,43.026178
2,2adb0c3f36bec31b,2,9.739511,1500,9.915722,6.580615,-3.158896,-48.003048,48.003048
3,8238992843699b54,2,11.120745,1500,9.951648,6.580615,-4.540130,-68.992485,68.992485
4,450ad86976e1cd6b,2,7.571902,1500,9.599694,6.580615,-0.991286,-15.063732,15.063732
...,...,...,...,...,...,...,...,...,...
3216,fb8dcc0fd37da388,2,9.964565,1500,8.880553,6.580615,-3.383950,-51.423003,51.423003
3217,4556499a052a931f,2,13.718119,1500,3.370890,6.580615,-7.137504,-108.462560,108.462560
3218,3a5697d1bc6df097,2,10.593429,1500,8.746059,6.580615,-4.012814,-60.979315,60.979315
3219,e9c17bbaf0165635,2,11.990226,1500,8.765720,6.580615,-5.409611,-82.205240,82.205240


In [26]:
df.energy_deviation_pct_abs.describe()

count    3221.000000
mean       32.371339
std        26.850392
min         0.000000
25%        14.027194
50%        25.499506
75%        39.886254
max       154.537666
Name: energy_deviation_pct_abs, dtype: float64

## linear modeling stuff

In [461]:
df_mean_dev.columns

Index(['cpu_ms', 'sampling_ratio', 'mean_energy_deviation',
       'std_energy_deviation', 'conf_int_95', 'recip_sampling_ratio'],
      dtype='object')

In [462]:
# define x & y
X = df_mean_dev.drop(['sampling_ratio', 'mean_energy_deviation',
       'std_energy_deviation', 'conf_int_95'], axis=1).query("cpu_ms <= 1000.0")
y = df_mean_dev.query("cpu_ms <= 1000.0")['mean_energy_deviation']
#X,y

In [463]:
# create train/test split of data, 80/20 split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 47)

In [464]:
mlr = LinearRegression()  
mlr.fit(x_train, y_train)
# output model coefficients
print("Coefficients: ", list(zip(X, mlr.coef_)), "\n Intercept: ", mlr.intercept_)

Coefficients:  [('cpu_ms', np.float64(-0.007500033309187127)), ('recip_sampling_ratio', np.float64(27.74706118498039))] 
 Intercept:  12.10948180092405


In [426]:
# make predictions based on test data
y_pred = mlr.predict(x_test)

In [427]:
# calculate rmse
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("The RMSE is: ", rmse)
print("The R^2 is: ", r2)

The RMSE is:  2.6228721344343486
The R^2 is:  0.9165208207097248


#### Plot some of the fits and original data

In [428]:
mlr.coef_, mlr.intercept_
# make function for using regression model
def recip_model(cpu_ms, recip_ratio):
    return (mlr.intercept_ + mlr.coef_[0]*cpu_ms + mlr.coef_[1]*recip_ratio)

### Random Forest Regression

In [502]:
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

rf_regressor.fit(x_train, y_train)

y_pred = rf_regressor.predict(x_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

single_data = x_test.iloc[0].values.reshape(1, -1)
predicted_value = rf_regressor.predict(single_data)
print(f"Predicted Value: {predicted_value[0]:.2f}")
print(f"Actual Value: {y_test.iloc[0]:.2f}")

print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Predicted Value: 25.87
Actual Value: 21.48
Root Mean Squared Error: 3.77
R-squared Score: 0.89


/Users/tedscott/anaconda3/envs/E2E/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


#### RF is not better

In [503]:
single_data

array([[0.5]])

### Quick out-of-sample tests

In [504]:
test_data_point = [[0.33]]
test_data_point

[[0.33]]

In [505]:
predicted_value = rf_regressor.predict(test_data_point)
print(f"RF Predicted Value: {predicted_value[0]:.2f}")

RF Predicted Value: 18.70


/Users/tedscott/anaconda3/envs/E2E/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [506]:
# MLR2 prediction for that point
y_pred = mlr2.predict(test_data_point)
print(f"MLR Predicted Value: {y_pred[0]:.2f}")

MLR Predicted Value: 19.54


/Users/tedscott/anaconda3/envs/E2E/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
